In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
!pip install openpyxl

In [4]:
#loading sheets
df1 = pd.read_excel('online_retail_II.xlsx', sheet_name='Year 2009-2010')
df2 = pd.read_excel('online_retail_II.xlsx', sheet_name='Year 2010-2011')

In [5]:
#changing invoice columns datatype to string since some cancellations invoice starts with "C"
df1['Invoice'] = df1['Invoice'].astype(str)
df2['Invoice'] = df2['Invoice'].astype(str)

In [6]:
#combining both dataframes
df = pd.concat([df1, df2], ignore_index=True)

In [7]:
print(df.shape)

(1067371, 8)


In [8]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [9]:
#Cleaning

In [10]:
df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  str           
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(2), str(2)
memory usage: 65.1+ MB


Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [11]:
#identifying cancelled orders
df['Is_Cancelled'] = df['Invoice'].str.startswith('C')
print(df['Is_Cancelled'].value_counts())

Is_Cancelled
False    1047877
True       19494
Name: count, dtype: int64


In [12]:
#removing missing customers IDs
df = df.dropna(subset=['Customer ID'])
print(df.shape)

(824364, 9)


In [13]:
#checking and removing duplicate rows
print(df.duplicated().sum())
df = df.drop_duplicates()
print(df.shape)

26479
(797885, 9)


In [14]:
#checking -ve quantity and price
print(df[df['Quantity'] <= 0]['Is_Cancelled'].value_counts())
print(df[df['Price'] <= 0].shape)

Is_Cancelled
True    18390
Name: count, dtype: int64
(70, 9)


In [15]:
df[df['Price'] <= 0].head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Is_Cancelled
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126.0,United Kingdom,False
6781,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.0,15658.0,United Kingdom,False
16107,490727,M,Manual,1,2009-12-07 16:38:00,0.0,17231.0,United Kingdom,False
18738,490961,22065,CHRISTMAS PUDDING TRINKET POT,1,2009-12-08 15:25:00,0.0,14108.0,United Kingdom,False
18739,490961,22142,CHRISTMAS CRAFT WHITE FAIRY,12,2009-12-08 15:25:00,0.0,14108.0,United Kingdom,False
32916,492079,85042,ANTIQUE LILY FAIRY LIGHTS,8,2009-12-15 13:49:00,0.0,15070.0,United Kingdom,False
40101,492760,21143,ANTIQUE GLASS HEART DECORATION,12,2009-12-18 14:22:00,0.0,18071.0,United Kingdom,False
47126,493761,79320,FLAMINGO LIGHTS,24,2010-01-06 14:54:00,0.0,14258.0,United Kingdom,False
48342,493899,22355,"CHARLOTTE BAG , SUKI DESIGN",10,2010-01-08 10:43:00,0.0,12417.0,Belgium,False
57619,494607,21533,RETRO SPOT LARGE MILK JUG,12,2010-01-15 12:43:00,0.0,16858.0,United Kingdom,False


In [16]:
#keeping only the rows where price>0, since these rows can skew metrics like average
df = df[df['Price'] > 0]
print(df.shape)

(797815, 9)


In [17]:
#new col- revenue
df['Revenue'] = df['Quantity'] * df['Price']
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Is_Cancelled,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,30.0


In [18]:
#finding month/year from invoice date for seasonal trend analysis
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')
df[['InvoiceDate', 'Year', 'Month', 'YearMonth']].head()

,InvoiceDate,Year,Month,YearMonth
0,2009-12-01 07:45:00,2009,12,2009-12
1,2009-12-01 07:45:00,2009,12,2009-12
2,2009-12-01 07:45:00,2009,12,2009-12
3,2009-12-01 07:45:00,2009,12,2009-12
4,2009-12-01 07:45:00,2009,12,2009-12


In [19]:
#checking outliers
df['Quantity'].describe()

count    797815.000000
mean         12.585594
std         191.163061
min      -80995.000000
25%           2.000000
50%           5.000000
75%          12.000000
max       80995.000000
Name: Quantity, dtype: float64

In [20]:
Q1 = df['Quantity'].quantile(0.25)
Q3 = df['Quantity'].quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + 1.5 * IQR

df['Is_Outlier'] = df['Quantity'] > upper_limit
print(df['Is_Outlier'].value_counts())

Is_Outlier
False    746696
True      51119
Name: count, dtype: int64


In [21]:
#standardizing country col
print(df['Country'].unique())

<StringArray>
[      'United Kingdom',               'France',                  'USA',
              'Belgium',            'Australia',                 'EIRE',
              'Germany',             'Portugal',                'Japan',
              'Denmark',          'Netherlands',               'Poland',
                'Spain',      'Channel Islands',                'Italy',
               'Cyprus',               'Greece',               'Norway',
              'Austria',               'Sweden', 'United Arab Emirates',
              'Finland',          'Switzerland',          'Unspecified',
              'Nigeria',                'Malta',                  'RSA',
            'Singapore',              'Bahrain',             'Thailand',
               'Israel',            'Lithuania',          'West Indies',
                'Korea',               'Brazil',               'Canada',
              'Iceland',              'Lebanon',         'Saudi Arabia',
       'Czech Republic',   'European 

In [22]:
print(df.shape)
df.info()

(797815, 14)
<class 'pandas.DataFrame'>
Index: 797815 entries, 0 to 1067370
Data columns (total 14 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Invoice       797815 non-null  str           
 1   StockCode     797815 non-null  object        
 2   Description   797815 non-null  object        
 3   Quantity      797815 non-null  int64         
 4   InvoiceDate   797815 non-null  datetime64[us]
 5   Price         797815 non-null  float64       
 6   Customer ID   797815 non-null  float64       
 7   Country       797815 non-null  str           
 8   Is_Cancelled  797815 non-null  bool          
 9   Revenue       797815 non-null  float64       
 10  Year          797815 non-null  int32         
 11  Month         797815 non-null  int32         
 12  YearMonth     797815 non-null  period[M]     
 13  Is_Outlier    797815 non-null  bool          
dtypes: bool(2), datetime64[us](1), float64(3), int32(2), int64(1), object(

In [23]:
#exporting processed sheet
df.to_csv('processed_data.csv', index=False)

In [24]:
df.head(10000).to_csv('processed_data_sample.csv', index=False)

In [25]:
#Insight 1: Top products by revenue
top_products = df.groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(10)
print(top_products)

Description
REGENCY CAKESTAND 3 TIER              261110.95
WHITE HANGING HEART T-LIGHT HOLDER    237678.61
JUMBO BAG RED RETROSPOT               132180.02
ASSORTED COLOUR BIRD ORNAMENT         123631.87
POSTAGE                               110338.51
PARTY BUNTING                         102089.38
PAPER CHAIN KIT 50'S CHRISTMAS         75388.48
CHILLI LIGHTS                          68453.50
JUMBO BAG STRAWBERRY                   63615.53
BLACK RECORD COVER FRAME               63009.83
Name: Revenue, dtype: float64


In [26]:
#Insight 2: Top countries by revenue
top_countries = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10)
print(top_countries)

Country
United Kingdom    1.348251e+07
EIRE              5.735098e+05
Netherlands       5.483307e+05
Germany           4.119592e+05
France            3.200463e+05
Australia         1.664444e+05
Switzerland       9.877941e+04
Spain             9.101344e+04
Sweden            8.742152e+04
Denmark           6.445959e+04
Name: Revenue, dtype: float64


In [27]:
#Insight 3: Top customers by revenue (for RFM/Pareto check)
top_customers = df.groupby('Customer ID')['Revenue'].sum().sort_values(ascending=False).head(10)
print(top_customers)

total_revenue = df['Revenue'].sum()
top_20_percent_customers = int(df['Customer ID'].nunique() * 0.2)
top_20_revenue = df.groupby('Customer ID')['Revenue'].sum().sort_values(ascending=False).head(top_20_percent_customers).sum()
print(f"\nTotal Revenue: {total_revenue}")
print(f"Top 20% customers count: {top_20_percent_customers}")
print(f"Revenue from top 20% customers: {top_20_revenue}")
print(f"Percentage: {(top_20_revenue/total_revenue)*100:.2f}%")

Customer ID
18102.0    570380.61
14646.0    523342.07
14156.0    296063.44
14911.0    265757.91
17450.0    231390.55
13694.0    190020.84
17511.0    168491.62
12415.0    143269.29
16684.0    141502.25
15061.0    124961.98
Name: Revenue, dtype: float64

Total Revenue: 16289991.288
Top 20% customers count: 1187
Revenue from top 20% customers: 12604499.071000002
Percentage: 77.38%


In [28]:
#Insight 4: Monthly/Seasonal trend
monthly_revenue = df.groupby('YearMonth')['Revenue'].sum()
print(monthly_revenue)

YearMonth
2009-12     660125.100
2010-01     530436.512
2010-02     487596.426
2010-03     633419.311
2010-04     558007.832
2010-05     557873.390
2010-06     568784.550
2010-07     560885.330
2010-08     585259.460
2010-09     778520.051
2010-10     961520.740
2010-11    1129025.162
2010-12     552372.860
2011-01     473731.900
2011-02     435534.070
2011-03     578576.210
2011-04     425222.671
2011-05     647011.670
2011-06     606862.520
2011-07     573112.321
2011-08     615078.090
2011-09     929356.232
2011-10     973306.380
2011-11    1126815.070
2011-12     341557.430
Freq: M, Name: Revenue, dtype: float64


In [29]:
#Insight 5: Cancellation analysis
cancel_by_country = df[df['Is_Cancelled']==True].groupby('Country')['Revenue'].sum().sort_values().head(10)
print(cancel_by_country)

total_cancelled = df['Is_Cancelled'].sum()
total_transactions = len(df)
print(f"\nCancellation rate: {(total_cancelled/total_transactions)*100:.2f}%")

Country
United Kingdom   -906729.83
EIRE              -43060.78
France            -28722.70
Norway            -20866.59
Spain             -17319.05
Germany           -13060.55
Singapore         -12158.90
Netherlands        -5707.39
Portugal           -4879.85
Denmark            -4121.10
Name: Revenue, dtype: float64

Cancellation rate: 2.31%


In [30]:
df[df['Year']==2011][df['Month']==12]['InvoiceDate'].max()

C:\Users\91959\AppData\Local\Temp\ipykernel_14148\3354868624.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df[df['Year']==2011][df['Month']==12]['InvoiceDate'].max()


Timestamp('2011-12-09 12:50:00')

In [32]:
df_sample = df.sample(n=10000, random_state=42)
df_sample.to_csv('processed_data_sample2.csv', index=False)